In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast

In [ ]:
df = pd.read_csv('Dataset.csv ')
columns_to_drop = ['closed_datetime', 'description', 'action_taken_timestamp']
df = df.drop(columns=columns_to_drop, errors='ignore')
df = df.dropna(subset=['latitude', 'longitude'])


In [ ]:
def clean_violation_string(val):
    try:
        val_list = ast.literal_eval(val)
        if isinstance(val_list, list) and len(val_list) > 0:
            return val_list[0].split(' [')[0].strip()
    except (ValueError, SyntaxError):
        return val
    return val

if 'violation_type' in df.columns:
    df['clean_violation'] = df['violation_type'].apply(clean_violation_string)
    df = df.drop(columns=['violation_type'])
    df = pd.get_dummies(df, columns=['clean_violation'], prefix='viol', dtype=int)


if 'validation_timestamp' in df.columns:
    df['is_validated'] = df['validation_timestamp'].notna().astype(int)



def extract_cyclical_features(df, col_name, prefix):
    """Safely extracts dates, times, and cyclical math from any datetime column."""
    if col_name in df.columns:
        
        df[col_name] = pd.to_datetime(df[col_name], errors='coerce')
        
        # UI Features (Standard Strings)
        df[f'{prefix}_date'] = df[col_name].dt.date
        df[f'{prefix}_time'] = df[col_name].dt.time
        
       
        hour = df[col_name].dt.hour
        minute = df[col_name].dt.minute
        day_of_week = df[col_name].dt.dayofweek
        month = df[col_name].dt.month
        
       
        df[f'{prefix}_is_weekend'] = (day_of_week >= 5).astype('Int64')

        
        time_in_hours = hour + (minute / 60.0)
        df[f'{prefix}_time_sin'] = np.sin(2 * np.pi * time_in_hours / 24.0)
        df[f'{prefix}_time_cos'] = np.cos(2 * np.pi * time_in_hours / 24.0)
        
        df[f'{prefix}_day_sin'] = np.sin(2 * np.pi * day_of_week / 7.0)
        df[f'{prefix}_day_cos'] = np.cos(2 * np.pi * day_of_week / 7.0)
        
        df[f'{prefix}_month_sin'] = np.sin(2 * np.pi * month / 12.0)
        df[f'{prefix}_month_cos'] = np.cos(2 * np.pi * month / 12.0)
        
    return df


df = df.dropna(subset=['created_datetime'])

df = extract_cyclical_features(df, 'created_datetime', 'created')
df = extract_cyclical_features(df, 'modified_datetime', 'modified')
df = extract_cyclical_features(df, 'validation_timestamp', 'val')


if 'modified_datetime' in df.columns and 'created_datetime' in df.columns:
    df['processing_delay_hours'] = ((df['modified_datetime'] - df['created_datetime']).dt.total_seconds() / 3600.0).round(2)


final_drop_columns = ['created_datetime', 'modified_datetime', 'validation_timestamp','val_date','val_time','created_time','created_date','modified_date','modified_time']
df = df.drop(columns=final_drop_columns, errors='ignore')

val_cyclical_columns = [
    'val_time_sin', 'val_time_cos', 
    'val_day_sin', 'val_day_cos', 
    'val_month_sin', 'val_month_cos'
]

for col in val_cyclical_columns:
    if col in df.columns:
        df[col] = df[col].fillna(-2.0)
        

if 'val_is_weekend' in df.columns:
    df['val_is_weekend'] = df['val_is_weekend'].fillna(-1)

In [ ]:
if 'updated_vehicle_number' in df.columns:
    df['updated_vehicle_number'] = df['updated_vehicle_number'].fillna('NOT_UPDATED')

if 'updated_vehicle_type' in df.columns:
    
    df['updated_vehicle_type'] = df['updated_vehicle_type'].str.upper().str.strip()
    df['updated_vehicle_type'] = df['updated_vehicle_type'].fillna('UNKNOWN')
    
    
    df = pd.get_dummies(df, columns=['updated_vehicle_type'], prefix='veh', dtype=int)

if 'validation_status' in df.columns:
    
    df['validation_status'] = df['validation_status'].str.upper().str.strip()
    
    df['validation_status'] = df['validation_status'].fillna('PENDING')
    
    
    df = pd.get_dummies(df, columns=['validation_status'], prefix='status', dtype=int)

In [ ]:
output_filename = 'Cleaned_Data.csv'
df.to_csv(output_filename, index=False)